# Notebook 2 — Concept Extraction

Extracts residual stream hidden states at `L_det` for all 1,879 breast SNOMED concepts using two co-primary methods, and produces the SNOMED→OMOP concept ID mapping required by Block B of notebook 3.

**Inputs**
- `src/config.py` → `L_DET` (set after running notebook 1)
- `data/embeddings-concept-openai/concepts.csv` — 1,879 breast concepts
- `data/athena/CONCEPT.csv` — Athena OMOP CDM vocabulary table (set `ATHENA_CONCEPT_CSV` below)

**Outputs**
- `data/stage1/hidden_states_last_token.npy` — shape `(n_concepts, d_model)`
- `data/stage1/hidden_states_mean_pool.npy`  — shape `(n_concepts, d_model)`
- `data/stage1/snomed_omop_map.csv`           — SNOMED CT ID → OMOP concept_id

**Blocked by:** `1-layer-calibration.ipynb` (`L_DET` must be set in `src/config.py`)

In [ ]:
# parameters
DATA_DIR           = "../../data/stage1"
CONCEPTS_CSV       = "../../data/embeddings-concept-openai/concepts.csv"
ATHENA_CONCEPT_CSV = "../../data/common/Athena/target/CONCEPT.csv"   # symlink → active Athena vocabulary

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from transformer_lens import HookedTransformer

sys.path.insert(0, "../..")
from src.config import MODEL_NAME, L_DET, TORCH_DTYPE

os.makedirs(DATA_DIR, exist_ok=True)

if L_DET is None:
    raise ValueError(
        "L_DET is None in src/config.py. "
        "Run 1-layer-calibration.ipynb first, then set L_DET = <value> in src/config.py."
    )

print(f"Model:  {MODEL_NAME}")
print(f"L_DET:  {L_DET}")
print(f"dtype:  {TORCH_DTYPE}")

In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME, dtype=TORCH_DTYPE)
model.eval()
print(f"d_model: {model.cfg.d_model}")

In [ ]:
concepts_df = pd.read_csv(CONCEPTS_CSV, dtype=str)
concept_names = concepts_df["preferred_term"].tolist()
n_concepts = len(concept_names)
d_model = model.cfg.d_model

print(f"Concepts: {n_concepts}")
print("Examples:", concept_names[:3])

## 2a. Extract hidden states at `L_det`

Two co-primary extraction methods run in a single forward pass per concept:
- **Method A** — last-token hidden state: `h[:, -1, :]`
- **Method B** — mean-pool over all token positions: `h.mean(dim=1)`

Only the target layer is cached (`names_filter`) to keep memory usage low.

In [ ]:
hook_name = f"blocks.{L_DET}.hook_resid_post"

hidden_last = np.zeros((n_concepts, d_model), dtype=np.float32)
hidden_mean = np.zeros((n_concepts, d_model), dtype=np.float32)

for i, concept in enumerate(concept_names):
    if i % 200 == 0:
        print(f"  {i} / {n_concepts}")

    tokens = model.to_tokens(concept)   # (1, n_tokens)

    with torch.no_grad():
        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name == hook_name
        )

    h = cache[hook_name][0]                    # (n_tokens, d_model)
    hidden_last[i] = h[-1].cpu().float().numpy()       # Method A
    hidden_mean[i] = h.mean(dim=0).cpu().float().numpy()  # Method B

print(f"Done. hidden_last shape: {hidden_last.shape}, hidden_mean shape: {hidden_mean.shape}")

In [ ]:
last_path = os.path.join(DATA_DIR, "hidden_states_last_token.npy")
mean_path = os.path.join(DATA_DIR, "hidden_states_mean_pool.npy")

np.save(last_path, hidden_last)
np.save(mean_path, hidden_mean)

print(f"Saved: {last_path}")
print(f"Saved: {mean_path}")

## 2b. SNOMED→OMOP mapping

Block B of `3-geometric-analysis.ipynb` uses KEEP embeddings, which are indexed by OMOP concept IDs. This step produces a mapping from the SNOMED CT IDs in `concepts.csv` to OMOP concept IDs using the Athena vocabulary table.

**Athena download:** https://athena.ohdsi.org — create a free account, search for SNOMED, and download `CONCEPT.csv`. Set `ATHENA_CONCEPT_CSV` in the parameters cell above.

Concepts without an OMOP match are excluded from Block B; the coverage rate is printed below.

In [ ]:
if not os.path.exists(ATHENA_CONCEPT_CSV):
    print(
        f"WARNING: {ATHENA_CONCEPT_CSV} not found.\n"
        "Download CONCEPT.csv from https://athena.ohdsi.org and set ATHENA_CONCEPT_CSV.\n"
        "Skipping SNOMED→OMOP mapping — Block B of notebook 3 will be unavailable."
    )
else:
    athena = pd.read_csv(ATHENA_CONCEPT_CSV, sep="\t", dtype=str, low_memory=False)
    snomed = athena[athena["vocabulary_id"] == "SNOMED"][["concept_code", "concept_id"]]
    print(f"SNOMED concepts in Athena: {len(snomed):,}")

    mapping = concepts_df.merge(
        snomed,
        left_on="concept_id",
        right_on="concept_code",
        how="left",
        suffixes=("_snomed", "_omop")
    )

    # Rename for clarity: concept_id_snomed = SNOMED CT ID, concept_id_omop = OMOP concept_id
    mapping = mapping.rename(columns={
        "concept_id": "snomed_concept_id",
        "concept_id_omop": "omop_concept_id"
    })[["snomed_concept_id", "omop_concept_id"]]

    mapped = mapping["omop_concept_id"].notna().sum()
    print(f"Mapped: {mapped} / {len(mapping)} concepts ({mapped/len(mapping)*100:.1f}%)")
    print(f"Unmapped (excluded from Block B): {len(mapping) - mapped}")

    out_path = os.path.join(DATA_DIR, "snomed_omop_map.csv")
    mapping.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")